# Phase 0: Fundamentals of DeepSeek-R1

## Understanding RL for Reasoning in LLMs

**Time Estimate:** 45 minutes  
**Prerequisites:** None - this is pure conceptual  
**No Code:** This phase focuses on building intuition

---

## What is DeepSeek-R1?

DeepSeek-R1 is a breakthrough paper that shows **how to teach language models to reason better using reinforcement learning (RL)**.

Instead of just prompting models to "think step by step," DeepSeek-R1 uses RL to *incentivize* longer, more deliberate reasoning chains that lead to correct answers.

### The Key Insight

**Before DeepSeek-R1:**
- We relied on supervised examples of reasoning (Chain-of-Thought)
- Models learned to mimic the patterns in our examples
- Limited by the quality and diversity of training examples

**With DeepSeek-R1:**
- We reward correct answers
- Models discover their own reasoning strategies
- More flexible, scalable, and powerful reasoning

---

## Part 1: Why Reasoning is Hard

### The Problem

Imagine a hard math problem:

> "Sarah has 12 apples. She gives half to Tom. Tom gives a third of his to Lisa. How many apples does Lisa have?"

A language model could answer:
1. **Instant answer:** "Lisa has 2 apples" (might be right by accident)
2. **Wrong path:** "Sarah gives half (6) to Tom. Tom gives half his apples (3) to Lisa." → Wrong!
3. **Right path:** "Sarah gives half (6) to Tom. Tom gives a third (2) to Lisa." → Correct!

The model needs to:
- Break down the problem
- Follow logical steps
- Not make arithmetic mistakes
- Reason about relationships

### Why Chain-of-Thought Helps

If we train the model on examples that *show* the step-by-step reasoning:
```
Q: Sarah has 12 apples...
A: Sarah gives 12/2 = 6 to Tom.
   Tom gives 6/3 = 2 to Lisa.
   Lisa has 2 apples.
```

The model learns to imitate this reasoning pattern.

### The Limitation

But we need:
- Millions of hand-labeled examples
- Coverage of all problem types
- Examples of different reasoning strategies
- Updates when we find better approaches

**This doesn't scale!** 📈

---

## Part 2: Reinforcement Learning Intuition

### What is RL?

RL is about learning by **reward and punishment**, not by imitation.

Think of teaching a dog:
- **Supervised Learning (SFT):** Show the dog how to sit → dog imitates
- **Reinforcement Learning:** Dog sits → give treat → dog learns sitting gets rewards

The key difference: The dog learns what *works*, not just what humans do.

### RL Components

1. **Agent** (our language model)
   - Takes actions (generates text)
   - Learns from rewards
   - Goal: maximize expected reward

2. **Environment** (the problem)
   - Presents problems
   - Evaluates solutions
   - Gives rewards

3. **Action** (what the model generates)
   - Reasoning steps
   - Intermediate answers
   - Final answer

4. **Reward** (signal of success)
   - +1 if answer is correct
   - 0 or -1 if answer is wrong
   - Can include intermediate rewards

### Simple Example

**Math Problem:** "What is 7 × 8?"

**Model generates (examples):**
```
Attempt 1: "The answer is 54"
  Reasoning quality: None
  Answer correctness: Wrong
  Reward: -1 ❌

Attempt 2: "7 × 8... let me think. 7 × 8 = 7 × (5 + 3) = 35 + 21 = 56. Correct!"
  Reasoning quality: Good
  Answer correctness: Correct
  Reward: +1 ✅

Attempt 3: "7 × 8 = 56"
  Reasoning quality: None
  Answer correctness: Correct
  Reward: +1 ✅ (but maybe less than Attempt 2)
```

The model learns:
- Showing work → higher reward
- Correct answer → gets reward
- Wrong answer → no reward

---

## Part 3: Understanding PPO (Proximal Policy Optimization)

### What is PPO?

PPO (Proximal Policy Optimization) is a popular RL algorithm that improves upon earlier methods. **PPO is the algorithm used in InstructGPT (ChatGPT) for Reinforcement Learning from Human Feedback (RLHF)!**

**Key insight:** PPO makes policy optimization more stable by limiting how much the policy can change in each update.

### The Problem PPO Solves

**Standard Policy Gradient (Simple but Unstable):**
```
If a move is good: Increase its probability
If a move is bad: Decrease its probability

But: How much should we change?
```

**The Danger:**
- Change too little: Learning is slow
- Change too much: Policy becomes unstable, might collapse

**Real Example:**
```
Game AI learning to play chess:

Bad update (too aggressive):
  "Move A won? Change probability from 0.1 to 0.9!"
  → Policy becomes extreme
  → Forgets other moves
  → Fails against new opponents

Good update (PPO's approach):
  "Move A won? Change probability from 0.1 to 0.15."
  → Gradual improvement
  → Keeps learning other moves
  → Stable, steady progress
```

### PPO's Solution: Trust Region

PPO's key idea is the **Trust Region**: "Only update the policy in a safe range."

```
Allowed change: probability ratio between 0.8 and 1.2
(or 1-ε to 1+ε where ε=0.2)

If ratio would go outside: Clip it!
```

**In Plain English:**
```
Old action probability: 10%
Advantage (is this good?): Very high

New probability would be: 50% (5x increase)
But PPO clips it to: 12% (1.2x increase)

This keeps learning stable!
```

### The PPO Algorithm (Step by Step)

#### Step 1: Collect Experience
```
Generate multiple attempts/trajectories
Get reward for each
```

#### Step 2: Calculate Advantage
```
Advantage = How good was this compared to average?
A(s,a) = (actual result) - (expected result)

Example:
  Expected value: 50
  Actual result: 70
  Advantage: +20 (good!)

  Expected value: 50
  Actual result: 30
  Advantage: -20 (bad!)
```

#### Step 3: PPO Update (The Key Innovation)
```
Calculate how much policy changed:
  ratio = P_new(action) / P_old(action)

Clip the ratio to stay safe:
  ratio = clip(ratio, 0.8, 1.2)
  (if ratio = 2.5, use 1.2 instead)

Compute loss:
  loss = -min(ratio × advantage, clipped_ratio × advantage)

Update policy:
  weights ← weights - learning_rate × gradient(loss)
```

### Why PPO's Clipping Matters

**Analogy: Learning to cook**

```
You made a dish, tasted it. Result: Delicious! (+100 points)

WITHOUT PPO (standard update):
  "Increase salt by 100x next time!"
  Result: Way too salty. Disaster.

WITH PPO (clipped update):
  "Increase salt by 20% next time."
  Result: Better! Gradual improvement.
```

### PPO in Practice (Why It's Popular)

**Advantages:**
1. ✅ Simple to understand
2. ✅ Stable training (clipping prevents wild swings)
3. ✅ Sample efficient (uses data well)
4. ✅ Works for many problems
5. ✅ Proven effective (used in ChatGPT)

**Key hyperparameters:**
- **ε (epsilon):** How much to limit changes (usually 0.1-0.2)
- **Learning rate:** How big updates are
- **Update epochs:** How many times to reuse collected data

### PPO Pseudocode

```
Initialize policy and value function

For each training iteration:
    1. Collect trajectories using current policy
    2. Calculate advantages: A = return - value_estimate
    3. For each update epoch:
        For each mini-batch of data:
            Calculate: ratio = π_new(a|s) / π_old(a|s)
            Clip ratio between (1-ε, 1+ε)
            Calculate loss: -min(ratio × A, clipped × A)
            Update policy parameters
    4. Update value function to predict returns better
```

### PPO Summary

| Aspect | Details |
|--------|----------|
| **What** | Policy gradient with clipped objective |
| **Why** | Stable, efficient RL training |
| **How** | Limit policy changes with clipping |
| **Used for** | RLHF in ChatGPT, InstructGPT |
| **Pros** | Simple, stable, sample-efficient |
| **Cons** | Still not perfect for all problems |
| **For LLMs** | Works but can be improved (→ GRPO) |

---

## Part 3.5: From PPO to GRPO

### Why Not Just Use PPO?

PPO works great for many things, but has some challenges for language models:

1. **Sample Inefficiency** - Needs many samples to learn well
2. **Variance Issues** - Rewards can be noisy
3. **Stability** - Large models can be unpredictable
4. **Compute Cost** - Very expensive for trillion-parameter models

### What GRPO Improves

GRPO (Group Relative Policy Optimization) takes PPO and optimizes it specifically for LLMs:

**GRPO's Innovation:**
Instead of comparing each output to a fixed baseline (PPO's approach), **compare outputs relative to each other in a group**.

**Why this matters:**
- Reduces noise in learning signal
- More stable training
- Better for reasoning tasks
- More sample efficient

---

## Part 4: Group Relative Policy Optimization (GRPO)

GRPO is DeepSeek-R1's core algorithm.

### Core Idea

Instead of comparing each output to a fixed baseline, compare outputs **relative to each other within a group**.

### Example

Model generates 4 attempts at a problem:

```
Attempt A: Answer = 56, Reasoning = "7×8=56" (short), Reward = 1
Attempt B: Answer = 56, Reasoning = "7×8... step by step... = 56" (long), Reward = 1  
Attempt C: Answer = 54, Reasoning = "Quick guess", Reward = 0
Attempt D: Answer = 56, Reasoning = "(no reasoning)", Reward = 1
```

**PPO's approach (fixed baseline):**
```
Value of state = 0.75 (average of all rewards)
All three correct answers (A, B, D) get same update
Problem: Doesn't distinguish between good and mediocre
```

**GRPO's approach (group relative):**
```
Group average reward: (1 + 1 + 0 + 1) / 4 = 0.75

Relative rewards:
  A: 1 - 0.75 = +0.25 (above average, basic approach)
  B: 1 - 0.75 = +0.25 (above average, shows work)
  C: 0 - 0.75 = -0.75 (below average, wrong)
  D: 1 - 0.75 = +0.25 (above average, no reasoning)

All correct answers still tied, but now we can add
process rewards to encourage B over A and D!
```

### Why GRPO is Better for Reasoning

1. **Clearer Signal** - Distinguishes between attempts better
2. **More Stable** - Comparing within groups reduces noise
3. **Sample Efficient** - Make best use of collected samples
4. **Encourages Reasoning** - Can reward process, not just outcome
5. **Designed for LLMs** - Handles text generation naturally

### GRPO vs PPO: Comparison

| Aspect | PPO | GRPO |
|--------|-----|------|
| **Baseline** | Fixed value function | Group mean |
| **Stability** | Good | Excellent |
| **Sample efficiency** | Good | Better |
| **Variance** | Medium | Low |
| **For LLMs** | Works | Optimized |
| **For reasoning** | Okay | Excellent |
| **Complexity** | Simple | Simple+ |
| **Used in** | ChatGPT (RLHF) | DeepSeek-R1 |

---

## Part 5: The DeepSeek-R1 Approach

### High-Level Process

```
1. Start with base LLM
   ↓
2. Define reward function (e.g., answer correctness)
   ↓
3. Generate multiple attempts per problem
   ↓
4. Calculate rewards for each attempt
   ↓
5. Use GRPO to update model weights
   → Encourage outputs with high relative reward
   → Discourage outputs with low relative reward
   ↓
6. Repeat until model improves
```

### Key Features

1. **Minimal Supervision**
   - Only need answer correctness (1-bit signal)
   - Don't need intermediate step labels
   - Scales to new domains easily

2. **Emergent Reasoning**
   - Model discovers reasoning strategies
   - Not forced into human-designed patterns
   - Can find novel approaches

3. **Scaling with Compute**
   - More reasoning → better performance
   - Longer chains during inference
   - Continues improving with larger models

4. **Broad Generalization**
   - Works across reasoning domains
   - Math, code, logic, planning, etc.
   - Transfers to unseen domains

---

## Part 6: Reward Function Design

### What Makes a Good Reward?

The reward function is crucial - it tells the model what success looks like.

#### Example 1: Math Problems
```python
def reward(attempt):
    if attempt.final_answer == ground_truth:
        return 1.0
    else:
        return 0.0
```
**Simple but effective:** Right answer = reward

#### Example 2: With Process Rewards
```python
def reward(attempt):
    # Reward for showing work
    reasoning_bonus = min(len(attempt.reasoning_steps) / 10, 0.2)
    
    # Reward for correctness
    correctness = 1.0 if attempt.final_answer == ground_truth else 0.0
    
    # Combined
    return correctness + (reasoning_bonus * correctness)
```
**More nuanced:** Reward reasoning + correctness

#### Example 3: Code Generation
```python
def reward(attempt):
    try:
        # Run the code, test against cases
        test_results = run_tests(attempt.code)
        return sum(test_results) / len(test_results)  # Fraction passing
    except:
        return 0.0  # Code doesn't even run
```
**Task-specific:** Tests passing = reward

### Design Principles

1. **Measure what matters** - Correctness first
2. **Make it differentiable** - Clear signal, not just binary
3. **Consider process** - Maybe reward intermediate steps
4. **Avoid gaming** - Can't be gamed to fake success
5. **Scale with complexity** - Harder problems → harder to get reward

---

## Part 7: Scaling Laws for Reasoning

### The Compute-Performance Trade-off

One of DeepSeek-R1's key findings:

**Longer reasoning chains → Better performance (but slower)**

```
Chain Length vs Accuracy

Quick Answer (1-2 steps):
  - Fast
  - 60% accuracy
  - Good for simple problems

Medium Chain (5-10 steps):
  - Balanced
  - 80% accuracy  ← Often optimal
  - Good for most problems

Long Chain (20+ steps):
  - Slow
  - 95% accuracy
  - Good for hard problems
```

### Scaling Laws Patterns

1. **Bigger models benefit more**
   - Small models: minimal gain from longer chains
   - Large models: significant gains from longer chains

2. **More compute during inference helps**
   - Spend computation at test time (longer chains)
   - Better than spending at train time
   - More flexible and adaptive

3. **Scaling continues**
   - Unlike supervised learning that plateaus
   - RL continues improving with more reasoning
   - Suggests room for future improvements

---

## Part 8: Key Insights & Concepts

### The Breakthrough

Why is DeepSeek-R1 important?

1. **Moves away from supervised learning** - Don't need labeled reasoning
2. **More sample efficient** - Learn from unlabeled data + rewards
3. **Better scaling** - Performance improves with model size
4. **Generalization** - Works across diverse reasoning tasks
5. **Interpretability** - Can see the reasoning chains

### New Terminology

**Policy** - The language model, viewed as a decision-making system
- Maps from problem → reasoning + answer
- Can be improved through RL

**Reward Signal** - The feedback telling the model if it's right
- Immediate: "answer is correct" → +1
- Intermediate: "step is correct" → +0.5
- Could be automatic: test execution, verification

**Group Relative Policy Optimization** - The training algorithm
- Generates multiple attempts
- Compares them relatively
- Updates the policy to favor better attempts

**Reasoning Chain / Chain-of-Thought** - The model's work
- Intermediate steps shown
- Can be long or short
- Emerges naturally from RL

---

## Part 9: Algorithm Hierarchy

### How They All Connect

```
Policy Gradients (Foundation)
    ↓
PPO (Proximal Policy Optimization)
  Used in: ChatGPT, InstructGPT
  Feature: Clipped objective for stability
    ↓
GRPO (Group Relative Policy Optimization)
  Used in: DeepSeek-R1
  Feature: Group-relative rewards for LLMs
    ↓
DeepSeek-R1
  Application: Teach LLMs to reason better
```

### Why Understanding This Hierarchy Matters

- **Policy Gradients** = Core mathematical principle
- **PPO** = Practical improvement (stability)
- **GRPO** = Further improvement for LLMs (relative rewards)
- **DeepSeek-R1** = Application to reasoning

Each builds on the previous one!

---

## Part 10: Putting It Together

### The Complete Picture

```
Problem: "How do we teach LLMs to reason?"

Traditional Answer (Supervised Learning):
  - Label examples with step-by-step reasoning
  - Supervised fine-tuning on these examples
  - Limitation: Scales poorly, limited by labels

DeepSeek-R1 Answer (Reinforcement Learning):
  1. Start with supervised pre-trained LLM
  2. Define reward: correctness of answer
  3. Use GRPO (improved PPO) for optimization
  4. Model discovers its own reasoning strategies
  5. Advantage: Scales better, more flexible, more powerful
```

### The Key Innovations

**PPO brought:**
- Stability through clipping
- Sample efficiency
- Foundation for RLHF in ChatGPT

**GRPO added:**
- Group-relative rewards (lower variance)
- Better for LLMs
- Better for reasoning tasks

**DeepSeek-R1 applied:**
- GRPO to reasoning in language models
- Shows continuous scaling with reasoning length
- Achieves SOTA on reasoning benchmarks

### Why This Matters

1. **For Researchers:** New approach to training, new research directions
2. **For Practitioners:** Can improve reasoning on their own tasks
3. **For the Field:** Suggests RL is key to AI improvement
4. **For Society:** Better reasoning in AI systems

---

## Summary

### Key Takeaways

✅ **Reasoning is hard** - Requires systematic problem decomposition  
✅ **RL can help** - Reward good reasoning, discourage bad  
✅ **PPO provides stability** - Clipped objective prevents wild policy changes  
✅ **GRPO optimizes for LLMs** - Group-relative rewards reduce variance  
✅ **Emergent behavior** - Models discover reasoning without explicit teaching  
✅ **Scales with compute** - More reasoning time → better results  
✅ **Generalizes broadly** - Works across reasoning domains  

### The Complete Algorithm Chain

```
Policy Gradients
      ↓ (Stability)
PPO (Clipped Updates)
      ↓ (For LLMs)
GRPO (Group Relative)
      ↓ (For Reasoning)
DeepSeek-R1
      ↓
You understand how it works! ✅
```

### Next Steps

Ready to implement? Move to **Phase 1: Basic Concepts**

There you'll implement:
- Basic RL concepts in Python
- Simple reward functions
- Policy comparison and improvement
- Hands-on experiments

---

## Questions to Ponder

Before moving on, think about:

1. **Why is reasoning hard?** What makes math problems different from classification?
2. **How would you reward reasoning?** What signal would you use for your own domain?
3. **Why does PPO use clipping?** What would happen without it?
4. **Why is GRPO better for LLMs?** What advantage does group-relative have?
5. **Where else could this apply?** Besides reasoning, what other LLM tasks could benefit?

---

## Further Reading

To deepen your understanding:
- Read Phase 1 for hands-on examples
- Review readme.md for related work
- Check rlhf_basic_primer.md for comparisons
- Reference prerequisites_reading_guide.md for paper recommendations

**You now understand the fundamentals!** Ready to code? 🚀